In [2]:
#import shutil
#shutil.rmtree("Regression_Output")

<h3>Table 1 — Daily Model Diagnostics</h3>

<div style="overflow-x:auto;">

| column | dtype | description |
|---|---|---|
| train_day | date | Training day |
| test_day | date | Evaluation day |
| symbol | string | Stock symbol |
| target | string | Target column (price measure + forecast horizon), e.g. `T_MidPrice_LogReturn_1s` |
| run_id | int | Identifier for features / preprocessing / model |
| n_train | int | Number of training observations |
| n_test | int | Number of test observations |
| mse | float | Model mean squared error |
| mse_bench | float | Benchmark (zero-return) mean squared error = `mean(y_test²)` |
| mse_ratio | float | `mse_bench / mse` |
| mae | float | Mean absolute error |
| r2_oos | float | Out-of-sample R² = `1 - mse / mse_bench` |

</div>

Planned (not yet computed): `directional_accuracy`, `directional_accuracy_bench` (= 0.5 for the zero-return benchmark), `residual_mean`, `residual_std`.

<hr>

<h3>Table 2 — Run-Specific Identifiers</h3>

<div style="overflow-x:auto;">

| run_id | model_version | feature_set_id | normalization_id |
|---|---|---|---|
| int | string | string | string |

</div>

<hr>

</div>


Add Table to store DM-related outputs. Something like:
Symbol, Trading Day, reg_model_ID, XGBoost_model_ID, LSTM_model_ID, d_reg_XBoost, d_reg_LSTM, ...

In [9]:
import os, warnings
from pathlib import Path
import time
from tqdm.auto import tqdm

import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression

import importlib
import utils.data_preprocessing as du
import utils.model_preparation as mu

importlib.reload(du)
importlib.reload(mu)

<module 'utils.model_preparation' from '/home/jovyan/_shared_storage/temp_stud/MA_Otto/utils/model_preparation.py'>

In [ ]:
SAVE_TICK_LEVEL_DATA = False

MODEL = "OLS"
MODEL_VERSION = "ols_v1"
FEATURE_SET_ID = "microP_l1Imb_9Lags"
NORMALIZATION_ID = ""

DATA_ROOT = "data/processed"
OUTPUT_ROOT = "Regression_Output"

In [ ]:
t = pd.read_parquet(f"{DATA_ROOT}/{du.SYMBOLS[0]}/{du.SAMPLE_DATES[0]}.parquet")

In [ ]:
# ============================================================
# Initialize Run
# ============================================================
warnings.filterwarnings("ignore")

RUN_ID = mu.initialize_run(
    output_root=OUTPUT_ROOT,
    model_version=MODEL_VERSION,
    feature_set_id=FEATURE_SET_ID,
    normalization_id=NORMALIZATION_ID
)

DIRS = mu.create_output_dirs(
    output_root=OUTPUT_ROOT,
    run_id=RUN_ID,
)

TICK_OUTPUT_DIR = DIRS["tick"]
DAILY_OUTPUT_DIR = DIRS["daily"]

daily_results = []

# Trading days used for the rolling walk-forward
SD = du.SAMPLE_DATES[:10]

sample_cols = pd.read_parquet(f"{DATA_ROOT}/{du.SYMBOLS[0]}/{SD[0]}.parquet").columns

FEATURE_COLS = list(sample_cols[sample_cols.str.startswith("F_")])
TARGET_COLS = list(sample_cols[sample_cols.str.startswith("T_")])

# ============================================================
# Main Loop
# ============================================================

GLOBAL_START = time.perf_counter()

# One symbol at a time: load all its days, then roll train(i) -> test(i+1).
# No feature/target normalization: OLS is scale-equivariant, so it does not
# affect fits or predictions. Revisit for regularized / NN models.
for symbol in tqdm(du.SYMBOLS[:-1], desc="Processing symbols", unit="symbol"):
    load_start = time.perf_counter()

    day_cache = {}

    for day in SD:
        df = pd.read_parquet(f"{DATA_ROOT}/{symbol}/{day}.parquet").sort_values("Timestamp")
        df = df.dropna(subset=FEATURE_COLS+TARGET_COLS)

        day_cache[day] = {
            "X": df[FEATURE_COLS].to_numpy(dtype=np.float32),
            "Y": df[TARGET_COLS].to_numpy(np.float32),

            "timestamp": df["Timestamp"].to_numpy(),
        }

    tqdm.write(f"{symbol}: Loaded all days in {time.perf_counter()-load_start:.2f}s")

    # --------------------------------------------------------
    # Rolling train/test
    # --------------------------------------------------------

    for i in tqdm(range(len(SD) - 1), desc=f"{symbol}", leave=False, unit="split"):

        train_day = SD[i]
        test_day = SD[i+1]

        train_cache = day_cache[train_day]
        test_cache = day_cache[test_day]

        X_train = train_cache["X"]
        X_test = test_cache["X"]

        Y_train = train_cache["Y"]
        Y_test = test_cache["Y"]

        n_train = X_train.shape[0]
        n_test = X_test.shape[0]

        model = LinearRegression()
        model.fit(X_train, Y_train)

        Y_pred = model.predict(X_test).astype(np.float32)

        # Residuals
        RESID = (Y_test - Y_pred).astype(np.float32)

        if SAVE_TICK_LEVEL_DATA:
            # Minimal tick-level store for Diebold-Mariano: Timestamp + one
            # residual column per target. y_true is NOT stored -- DM's loss
            # differential depends only on residuals, and y_true is already
            # available as the T_ columns in data/processed (join on Timestamp).
            #
            # float16 residuals: ~2x smaller than float32, aggregate MSE error
            # ~1e-6 (negligible for DM). feather+zstd is the smallest lossless
            # container here (parquet inflates incompressible floats).
            tick_df = pd.DataFrame(RESID.astype(np.float16), columns=TARGET_COLS)
            tick_df.insert(0, "Timestamp", test_cache["timestamp"])

            mu.save_table(
                df=tick_df,
                root_dir=TICK_OUTPUT_DIR,
                filename="tick_residuals.feather",
                partition_cols={
                    "date": test_day,
                    "symbol": symbol
                },
                file_format="feather",
                compression="zstd"
            )

        # Benchmark = zero-return (martingale) forecast: mse_bench = mean(Y_test^2).
        # Directional accuracy of this benchmark is undefined (constant 0 has no
        # sign); when directional metrics are added, record dir_bench = 0.5.
        mae = np.mean(np.abs(RESID), axis=0)
        mse = np.mean(RESID ** 2, axis=0)
        mse_bench = np.mean(Y_test ** 2, axis=0)

        mse_ratio = np.divide(
            mse_bench,
            mse,
            out=np.full_like(mse, np.nan),
            where=mse > 0
        )

        r2_oos = np.divide(
            mse,
            mse_bench,
            out=np.full_like(mse, np.nan),
            where=mse_bench > 0
        )

        r2_oos = 1.0 - r2_oos

        for j, target in enumerate(TARGET_COLS):
            daily_results.append({
                "train_day": train_day,
                "test_day": test_day,
                "symbol": symbol,
                "target": target,
                "run_id": RUN_ID,
                "n_train": int(n_train),
                "n_test": int(n_test),
                "mse": float(mse[j]),
                "mse_bench": float(mse_bench[j]),
                "mse_ratio": float(mse_ratio[j]),
                "mae": float(mae[j]),
                "r2_oos": float(r2_oos[j]),
            })

tqdm.write(f"\nTOTAL PIPELINE TIME: {time.perf_counter()-GLOBAL_START:.2f}s")
# ============================================================
# Save Outputs
# ============================================================

mu.save_table(
    df=pd.DataFrame(daily_results),
    root_dir=DAILY_OUTPUT_DIR,
    filename="daily_diagnostics.parquet",
    file_format="parquet"
)

In [ ]:
daily_out = pd.read_parquet(f"{DAILY_OUTPUT_DIR}/daily_diagnostics.parquet")
daily_out[daily_out["mse_ratio"] < 1]

In [ ]:
len(daily_out[daily_out["mse_ratio"]<1]) / len(daily_out)

In [ ]:
daily_out_1 = pd.read_parquet("Regression_Output/DailyDiagnostics/1/daily_diagnostics.parquet")
daily_out_1